<a href="https://colab.research.google.com/github/kazumasaiinuma/DS340-Project/blob/main/AIPhotoCulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U kaggle
import os, getpass
os.environ["KAGGLE_USERNAME"] = "kazumasai"   # the name in your kaggle.com/<username> URL
key = getpass.getpass("Paste Kaggle API key: ")
os.environ["KAGGLE_KEY"] = key
os.environ["KAGGLE_API_TOKEN"] = key
!kaggle datasets list -s triage | head -5

Paste Kaggle API key: ··········
ref                                                                              title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ilkeryildiz/emergency-service-triage-application                                 Emergency Service - Triage Application                    37064  2021-08-28 13:42:23.920000           6084         73                1  
jacopoferretti/emergency-triage-venomous-bites-dataset                           Emergency Triage: Venomous Bites Dataset               14455693  2026-06-20 09:27:25.403000            326         22                1  
maalona/hospital-triage-and-patient-history-data                                 Hospital Triag

In [2]:
!kaggle datasets download -d ericwolter/triage -p /content
!unzip -q /content/triage.zip "train_val/*" -d /content/data
!rm /content/triage.zip
!ls /content/data/train_val

Dataset URL: https://www.kaggle.com/datasets/ericwolter/triage
License(s): copyright-authors
100% 12.7G/12.7G [02:38<00:00, 86.0MB/s]

README.txt	  test.m	      train_val_imgs	    val_pairlist.txt
reviews_trainval  train_pairlist.txt  train_val_series.mat


In [3]:
!ls /content/data/train_val/train_val_imgs | wc -l
!wc -l /content/data/train_val/train_pairlist.txt /content/data/train_val/val_pairlist.txt

12988
 12075 /content/data/train_val/train_pairlist.txt
   483 /content/data/train_val/val_pairlist.txt
 12558 total


In [4]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ds340
!cp /content/data/train_val/*.txt /content/data/train_val/*.mat /content/drive/MyDrive/ds340/

Mounted at /content/drive


In [6]:
!pip install -q -U mediapipe opencv-python-headless
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

import cv2, glob, os, numpy as np, pandas as pd, mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
from tqdm import tqdm

IMG = "/content/data/train_val/train_val_imgs"
lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mpt.BaseOptions(model_asset_path="/content/face_landmarker.task"),
    output_face_blendshapes=True, num_faces=6, min_face_detection_confidence=0.5))

def lap_var(g):
    return cv2.Laplacian(g, cv2.CV_64F).var()

rows = []
for path in tqdm(sorted(glob.glob(f"{IMG}/*.JPG"))):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                             data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
    face_sharp, blink, smile = [], [], []
    for pts, bs in zip(res.face_landmarks, res.face_blendshapes):
        xs = [p.x for p in pts]
        ys = [p.y for p in pts]
        x0, x1 = max(0, int(min(xs) * w)), min(w, int(max(xs) * w))
        y0, y1 = max(0, int(min(ys) * h)), min(h, int(max(ys) * h))
        crop = gray[y0:y1, x0:x1]
        if crop.size > 400:
            face_sharp.append(lap_var(crop))
        b = {c.category_name: c.score for c in bs}
        blink.append(max(b["eyeBlinkLeft"], b["eyeBlinkRight"]))  # worst eye on this face
        smile.append((b["mouthSmileLeft"] + b["mouthSmileRight"]) / 2)
    s, p = os.path.basename(path)[:-4].split("-")
    rows.append(dict(series=int(s), photo=int(p), n_faces=len(blink),
                     face_sharp=max(face_sharp) if face_sharp else np.nan,
                     global_sharp=lap_var(gray),
                     mp_blink_max=max(blink) if blink else np.nan,   # anyone blinking?
                     mp_smile_mean=np.mean(smile) if smile else np.nan))

feat = pd.DataFrame(rows)
# Use face sharpness when there's a face, otherwise whole-image sharpness
feat["sharp"] = np.log1p(feat.face_sharp.fillna(feat.global_sharp))
feat.to_csv("/content/drive/MyDrive/ds340/features_v1.csv", index=False)
feat.describe()

100%|██████████| 12988/12988 [07:48<00:00, 27.73it/s]


,series,photo,n_faces,face_sharp,global_sharp,mp_blink_max,mp_smile_mean,sharp
count,12988.000000,12988.000000,12988.000000,1788.000000,12988.000000,1788.000000,1.788000e+03,12988.000000
mean,2844.751694,1.985371,0.191022,470.573478,1591.500851,0.268723,4.554925e-01,6.617813
std,1724.944417,1.076329,0.539719,508.911005,2079.071691,0.196973,3.417500e-01,1.301912
min,1.000000,1.000000,0.000000,9.612083,1.353272,0.001731,9.663835e-08,0.855807
25%,1349.000000,1.000000,0.000000,151.080126,329.388474,0.127922,5.221378e-02,5.755465
50%,2754.500000,2.000000,0.000000,318.257944,828.604371,0.219078,5.100099e-01,6.674430
75%,4338.000000,2.000000,0.000000,598.305679,2010.081900,0.362943,7.664139e-01,7.572653
max,5953.000000,8.000000,5.000000,4906.638152,23985.954208,0.978815,9.796981e-01,10.085265


In [7]:
cols = ["series", "p1", "p2", "ratio", "rank1", "rank2"]
val = pd.read_csv("/content/data/train_val/val_pairlist.txt", sep=r"\s+", header=None, names=cols)
s = feat.set_index(["series", "photo"]).sharp
val["pred"] = [s[(r.series, r.p1)] > s[(r.series, r.p2)] for r in val.itertuples()]
val["truth"] = val.ratio > 0.5
conf = (val.ratio >= 0.7) | (val.ratio <= 0.3)
print(f"All pairs:       {(val.pred == val.truth).mean():.3f}  (n={len(val)})")
print(f"Confident pairs: {(val.pred == val.truth)[conf].mean():.3f}  (n={conf.sum()})")
print(f"Images with a face: {(feat.n_faces > 0).mean():.1%}")

All pairs:       0.553  (n=483)
Confident pairs: 0.592  (n=292)
Images with a face: 13.8%
